# Major ETL for Ranch

In [4]:
import geopandas as gpd

gdf = gpd.read_file("data/cariboord_parcels_layer_1.geojson")

In [5]:
# key parcel number
numbers = [
    "011-822-091",
    "003-328-465",
    "013-375-105",
    "007-526-270",
    "013-081-241",
    "013-261-681",
    "013-261-711",
    "013-261-746",
    "013-375-130",
    "013-375-202",
    "013-375-211",
    "013-389-777",
    "013-366-599",
    "025-259-440",
    "019-044-437",
    "013-374-869",
    "013-374-885",
    "013-374-834",
    "013-261-771",
    "007-496-401",
    "004-293-380",
    "004-293-398",
    "013-312-308",
    "011-694-173",
    "013-314-581",
    "003-328-350",
    "009-871-969"
]

key_objectID = [
    23966,
    32663,
    24119,
    32201,
    28001,
    29993,
    23956,
    23967,
    23972,
    23962
]

key_parcels = gdf[gdf["Designation3"].isin(numbers) | gdf["OBJECTID"].isin(key_objectID)]
key_parcels.shape

(37, 33)

In [6]:
key_parcels.head()

,OBJECTID,GlobalID,Name,PlanID,PlanNumber,PIN,PID,Designation1,Designation2,Designation3,...,ModifiedDate,StatedArea,ErrorReportedBy,Municipality,Shape__Area,Shape__Length,Join_Count,TARGET_FID,Juroll,geometry
76,77,{911872A1-C142-414B-989C-AA1D7C79A466},011694173,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,6110171.0,11694173,SW4,2957,011-694-173,...,1515501033000,None,DATACOMPILATION,0163,627149.514632,3181.390264,1,77,2472741286000,"POLYGON ((-121.21085 51.72212, -121.21066 51.7..."
1034,1035,{1B4E771F-98A8-40D7-8730-C886A7CD9570},013366599,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,NaN,13366599,NE4,4185,013-366-599,...,1515500875000,None,DATACOMPILATION,0163,647680.736877,3219.144637,1,1035,2472741705006,"POLYGON ((-121.2343 51.69035, -121.24594 51.69..."
1112,1113,{3EDD98B8-789F-4155-8A81-04F379820023},013312308,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,NaN,13312308,NW4,2951,013-312-308,...,1515500830000,None,DATACOMPILATION,0163,623653.841862,3226.618280,1,1113,2472741276000,"POLYGON ((-121.20133 51.71208, -121.20091 51.7..."
3701,3702,{BAE0DA62-0A7B-48D6-97C9-0655DEF0CD49},004293380,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,6133340.0,4293380,722,,004-293-380,...,1547230504000,None,DATACOMPILATION,0163,323529.746844,2413.129789,1,3702,2472740488000,"POLYGON ((-121.2165 51.71126, -121.22262 51.71..."
3922,3923,{F1D45DDB-B44F-4123-A469-F073A50D9015},013261771,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,NaN,13261771,720,,013-261-771,...,1547230505000,None,DATACOMPILATION,0163,516882.675940,2923.009795,1,3923,2472740486000,"POLYGON ((-121.21861 51.72022, -121.21856 51.7..."


In [7]:
key_parcels.to_file("data/key_parcels.geojson", driver="GeoJSON")

# Four Hearts Separate Package Values CSV

In [8]:
import pandas as pd

df = pd.read_csv("data/FourHearts_Separate_package_values - Sheet1.csv")
df.head()
df.shape


(29, 14)

In [9]:
df.columns

Index(['key_parcel_char', 'key_parcel_num', 'DL#', 'Parcel ID',
       'Brief description', 'Acres', 'ALR status', 'Assesed Value Land',
       'Assesed Value Improve', 'Assesed Value TOTAL', 'BD List Value',
       'Estimated property transfer tax ()',
       'Estimated PTT (cultivated farmland)', 'Estimated PTT (rangeland)'],
      dtype='object')

In [10]:
# convert last 7 columns to float by removing commas
df.iloc[:, -7:] = df.iloc[:, -7:].replace(',', '', regex=True).astype(float)
# replace ‑ with - for parcel id
df["Parcel ID"] = df["Parcel ID"].str.replace('‑', '-')
df.head()

,key_parcel_char,key_parcel_num,DL#,Parcel ID,Brief description,Acres,ALR status,Assesed Value Land,Assesed Value Improve,Assesed Value TOTAL,BD List Value,Estimated property transfer tax (),Estimated PTT (cultivated farmland),Estimated PTT (rangeland)
0,C,13.0,4212 SE ¼,013-374-834,Ranch manager’s house & hay barn,160,Yes,10425.0,277200.0,287625.0,850000.0,18068.0,18068.0,15188.0
1,C,8.0,4185 NE ¼,013-366-599,no buildings,160,Yes,11165.0,NaN,11165.0,400000.0,12080.0,12080.0,9200.0
2,C,9.0,4211 BL. B,025-259-440,no buildings,156,Yes,24959.0,NaN,24959.0,450000.0,11728.0,11728.0,8920.0
3,C,10.0,4212 NE ¼,019-044-437,no buildings,160,Yes,10425.0,NaN,10425.0,500000.0,12080.0,12080.0,9200.0
4,C,11.0,4212 NW ¼,013-374-869,no buildings,160,Yes,10425.0,NaN,10425.0,500000.0,12080.0,12080.0,9200.0


## Setting Four Hearts Package Flag by PID


In [11]:
package_dict = {
    "FOUR HEARTS RANCH - THE 103 MILE RANCH": [
        "013-374-834",
        "013-366-599",
        "025-259-440",
        "019-044-437",
        "013-374-869",
        "013-374-885"
    ],
    "FOUR HEARTS RANCH - THE EQUESTRIAN ESTATE" : [
        "007-496-401",
        "013-312-308",
        "011-822-091",
        "013-261-771",
        "011-694-173",
        "004-293-398",
        "004-293-380",
        "009-871-969",
        "003-328-350"
    ],
    "THE SIMON LAKE RANCH" : [
        "013-261-711",
        "013-081-241",
        "013-389-777",
        "013-261-746"
    ],
    "THE MILCH HOMESTEAD RANCH" : [
        "013-314-581",
        "007-526-270",
        "013-261-681",
        "013-375-130",
        "013-375-202",
        "013-375-211"
    ],
    "THE EXETER PROPERTIES": [
        "013-375-105",
        "003-328-465"
    ]
}

# create a new column called "Four Hearts Package"
parcel_to_package = {}
for package_name, parcels in package_dict.items():
    for parcel in parcels:
        parcel_to_package[parcel] = package_name

# Create the new column based on the mapping
df["Four Hearts Package"] = df["Parcel ID"].map(parcel_to_package)
df.head()

,key_parcel_char,key_parcel_num,DL#,Parcel ID,Brief description,Acres,ALR status,Assesed Value Land,Assesed Value Improve,Assesed Value TOTAL,BD List Value,Estimated property transfer tax (),Estimated PTT (cultivated farmland),Estimated PTT (rangeland),Four Hearts Package
0,C,13.0,4212 SE ¼,013-374-834,Ranch manager’s house & hay barn,160,Yes,10425.0,277200.0,287625.0,850000.0,18068.0,18068.0,15188.0,FOUR HEARTS RANCH - THE 103 MILE RANCH
1,C,8.0,4185 NE ¼,013-366-599,no buildings,160,Yes,11165.0,NaN,11165.0,400000.0,12080.0,12080.0,9200.0,FOUR HEARTS RANCH - THE 103 MILE RANCH
2,C,9.0,4211 BL. B,025-259-440,no buildings,156,Yes,24959.0,NaN,24959.0,450000.0,11728.0,11728.0,8920.0,FOUR HEARTS RANCH - THE 103 MILE RANCH
3,C,10.0,4212 NE ¼,019-044-437,no buildings,160,Yes,10425.0,NaN,10425.0,500000.0,12080.0,12080.0,9200.0,FOUR HEARTS RANCH - THE 103 MILE RANCH
4,C,11.0,4212 NW ¼,013-374-869,no buildings,160,Yes,10425.0,NaN,10425.0,500000.0,12080.0,12080.0,9200.0,FOUR HEARTS RANCH - THE 103 MILE RANCH


In [12]:
# left join those 27 parcels with Parcel ID by using Parcel ID = Designation3 to the gdf
gdf_with_price = key_parcels[key_parcels["Designation3"].notna()].merge(
    df,
    how="left",
    left_on="Designation3",
    right_on="Parcel ID"
)

gdf_with_price = key_parcels.copy()

mask = gdf_with_price["Designation3"].notna()

gdf_with_price.loc[mask, df.columns] = (
    gdf_with_price[mask]
    .merge(
        df,
        how="left",
        left_on="Designation3",
        right_on="Parcel ID"
    )
    .set_index(gdf_with_price[mask].index)
    [df.columns]
)


In [13]:
license_milch = [23972, 23962, 23956, 23967] # License - "The Milch" ; THE MILCH HOMESTEAD RANCH
license_103 = [29993, 28001, 23966, 32663, 24119, 32201] # License - "The 103" ; FOUR HEARTS RANCH - THE 103 MILE RANCH

dl_map = {
    23962: "4198",
    23972: "4197",
    23956: "4229",
    23967: "4228",
    29993: "4221",
    28001: "4221",
    23966: "4221",
    32663: "4185",
    24119: "4185",
    32201: "4185"
}

# create dict 
for dl in license_103:
    gdf_with_price.loc[gdf_with_price["OBJECTID"] == dl, "Brief description"] = 'License - "The 103"'
    gdf_with_price.loc[gdf_with_price["OBJECTID"] == dl, "Four Hearts Package"] = "FOUR HEARTS RANCH - THE 103 MILE RANCH"
    
for dl in license_milch:
    gdf_with_price.loc[gdf_with_price["OBJECTID"] == dl, "Brief description"] = 'License - "The Milch"'
    gdf_with_price.loc[gdf_with_price["OBJECTID"] == dl, "Four Hearts Package"] = "THE MILCH HOMESTEAD RANCH"

# assign DL# to the parcels
for dl in dl_map:
    gdf_with_price.loc[gdf_with_price["OBJECTID"] == dl, "DL#"] = dl_map[dl]


In [14]:
# create new column called "License" if the DL# is in list(dl_map.values())
gdf_with_price["License"] = gdf_with_price["DL#"].isin(list(dl_map.values()))

## Handling ALR Status

In [15]:
gdf_with_price['ALR status'].fillna('Yes', inplace=True)

/var/folders/6h/jr32ngxd3497bhc38dpzkbzr0000gn/T/ipykernel_6165/718083219.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  gdf_with_price['ALR status'].fillna('Yes', inplace=True)


In [16]:
gdf_with_price.to_file("data/four_hearts_parcels_with_price.geojson", driver="GeoJSON")

In [17]:
gdf_with_price.columns

Index(['OBJECTID', 'GlobalID', 'Name', 'PlanID', 'PlanNumber', 'PIN', 'PID',
       'Designation1', 'Designation2', 'Designation3', 'ParcelStatus',
       'RegionalDistrict', 'ParcelClass', 'SourceParcelID', 'GeometrySource',
       'IsRemainder', 'ParcelStartDate', 'PositionalError', 'LegalDescription',
       'CaptureMethod', 'OwnerType', 'Compiled', 'CreatedDate', 'ModifiedDate',
       'StatedArea', 'ErrorReportedBy', 'Municipality', 'Shape__Area',
       'Shape__Length', 'Join_Count', 'TARGET_FID', 'Juroll', 'geometry',
       'key_parcel_char', 'key_parcel_num', 'DL#', 'Parcel ID',
       'Brief description', 'Acres', 'ALR status', 'Assesed Value Land',
       'Assesed Value Improve', 'Assesed Value TOTAL', 'BD List Value',
       'Estimated property transfer tax ()',
       'Estimated PTT (cultivated farmland)', 'Estimated PTT (rangeland)',
       'Four Hearts Package', 'License'],
      dtype='object')

In [18]:
# for each Four Hearts Package, create a new polygon union in gdf with all the parcels in that package, save it separately and with that package name for joining later
records = []

for package_name, parcels in package_dict.items():
    package_parcels = gdf_with_price[
        gdf_with_price["Four Hearts Package"] == package_name
    ]

    if package_parcels.empty:
        continue

    union_geom = package_parcels.geometry.union_all()

    records.append({
        "package_name": package_name,
        "geometry": union_geom
    })

# Create GeoDataFrame
packages_gdf = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs=gdf_with_price.crs
)


In [19]:
packages_gdf.to_file("data/four_hearts_packages.geojson", driver="GeoJSON")

# Full Ranch Boundary

In [20]:
big_polygon = packages_gdf.unary_union
ranch_gdf = gpd.GeoDataFrame(geometry=[big_polygon], crs=packages_gdf.crs)
# add some name column
ranch_gdf["Name"] = "FOUR HEARTS RANCH"
# area column
ranch_gdf["Area"] = ranch_gdf.geometry.area
# area in acres
ranch_gdf["Acres"] = ranch_gdf["Area"] * 0.000247105
ranch_gdf.to_file("data/four_hearts_ranch_boundary.geojson", driver="GeoJSON")
# KML File (coordinate point explore)

/var/folders/6h/jr32ngxd3497bhc38dpzkbzr0000gn/T/ipykernel_6165/523405260.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  big_polygon = packages_gdf.unary_union
/var/folders/6h/jr32ngxd3497bhc38dpzkbzr0000gn/T/ipykernel_6165/523405260.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  ranch_gdf["Area"] = ranch_gdf.geometry.area


In [21]:
ranch_gdf

,geometry,Name,Area,Acres
0,"POLYGON ((-121.28022 51.70477, -121.26921 51.7...",FOUR HEARTS RANCH,0.003313,8.186692e-07


# KML File (coordinate point explore)

In [22]:
import geopandas as gpd

def load_kmz_reliably(kmz_path):
    # Use the 'vsizip' prefix to tell the engine to look inside the zip
    # We point directly to the doc.kml inside the archive
    vsi_path = f"/vsizip/{kmz_path}/doc.kml"
    
    try:
        # engine='pyogrio' is significantly more robust for KML
        gdf = gpd.read_file(vsi_path, engine='pyogrio')
        return gdf
    except Exception as e:
        return f"Still failing: {e}"

# Usage
path = "qgis_file/Earth_FOUR HEARTS RANCH.kmz"
ranch_gdf = load_kmz_reliably(path)

if isinstance(ranch_gdf, gpd.GeoDataFrame):
    print("Success! Data loaded.")
    print(ranch_gdf.head())

Success! Data loaded.
               Name Description                         geometry
0    EQUIPMENT SHED              POINT Z (-121.21044 51.71833 0)
1              SHOP              POINT Z (-121.21166 51.71804 0)
2  Wrangler's Cabin              POINT Z (-121.21489 51.71832 0)
3          PADDOCKS               POINT Z (-121.2153 51.71745 0)
4             ARENA              POINT Z (-121.21001 51.71788 0)


/opt/miniconda3/envs/mapperia_env/lib/python3.10/site-packages/pyogrio/geopandas.py:275: UserWarning: More than one layer found in 'doc.kml': 'THE EQUESTRIAN CENTRE' (default), 'THE EQUESTRIAN ESTATE', 'FOUR HEARTS RANCH'. Specify layer parameter to avoid this warning.
  result = read_func(


In [23]:
import geopandas as gpd
from pyogrio import list_layers

path = "qgis_file/Earth_FOUR HEARTS RANCH.kmz"
vsi_path = f"/vsizip/{path}/doc.kml"

# List all internal layers
layers = list_layers(vsi_path)
print("Layers found:", layers)

Layers found: [['THE EQUESTRIAN CENTRE' 'Point Z']
 ['THE EQUESTRIAN ESTATE' 'Unknown']
 ['FOUR HEARTS RANCH' 'Unknown']]


In [24]:
import geopandas as gpd
import pandas as pd
from pyogrio import list_layers
from shapely.ops import transform

def load_ranch_kmz(kmz_path):
    vsi_path = f"/vsizip/{kmz_path}/doc.kml"
    
    # list_layers returns (array_of_names, array_of_types)
    layer_infos = list_layers(vsi_path)
    print(layer_infos)
    
    all_gdfs = []
    for layer_names in layer_infos:
        for name in layer_names:
            try:
                # Read the layer by name
                gdf = gpd.read_file(vsi_path, engine='pyogrio', layer=name)
                
                if not gdf.empty:
                    # Flatten for Folium/Streamlit compatibility (2D)
                    gdf.geometry = gdf.geometry.map(
                        lambda geom: transform(lambda x, y, z=None: (x, y), geom) 
                        if geom is not None else None
                    )
                    
                    gdf['kmz_folder'] = name
                    all_gdfs.append(gdf)
            except Exception as e:
                print(f"Skipping {name}: {e}")

    if not all_gdfs:
        return None

    return pd.concat(all_gdfs, ignore_index=True)

# Run it
path = "qgis_file/Earth_FOUR HEARTS RANCH.kmz"
ranch_gdf = load_ranch_kmz(path)

# Verify the Points and Boundaries
print(ranch_gdf[['Name', 'kmz_folder', 'geometry']].head())

# only get the points
points_gdf = ranch_gdf[ranch_gdf.geometry.geom_type == 'Point']



[['THE EQUESTRIAN CENTRE' 'Point Z']
 ['THE EQUESTRIAN ESTATE' 'Unknown']
 ['FOUR HEARTS RANCH' 'Unknown']]
Skipping Point Z: Layer 'Point Z' could not be opened
Skipping Unknown: Layer 'Unknown' could not be opened
Skipping Unknown: Layer 'Unknown' could not be opened
               Name             kmz_folder                     geometry
0    EQUIPMENT SHED  THE EQUESTRIAN CENTRE  POINT (-121.21044 51.71833)
1              SHOP  THE EQUESTRIAN CENTRE  POINT (-121.21166 51.71804)
2  Wrangler's Cabin  THE EQUESTRIAN CENTRE  POINT (-121.21489 51.71832)
3          PADDOCKS  THE EQUESTRIAN CENTRE   POINT (-121.2153 51.71745)
4             ARENA  THE EQUESTRIAN CENTRE  POINT (-121.21001 51.71788)


## Map Pin styling

In [25]:
import re

def generate_calcite_url(row):
    # Ensure name is a string and lowercase for matching
    name = str(row['Name']).lower()
    
    # Mapping Dictionary {Regex Pattern: Calcite Icon Name}
    patterns = {
        r"barn|equestrian|paddock|arena|pen": "barn",
        r"shop": "workshop",
        r"house|cabin|outpost|estate|ranch": "house",
        r"lake": "ocean",
        r"shed": "box-van",
        r"airstrip|hangar": "airplane",
        r"gate": "entrance"
    }
    
    # Default fallback icon
    selected_icon = "pins" 
    
    for pattern, icon in patterns.items():
        if re.search(pattern, name):
            selected_icon = icon
            break
            
    return f"https://raw.githubusercontent.com/Esri/calcite-point-symbols/refs/heads/master/icons/{selected_icon}-21.svg"


points_gdf['map_pin_icon'] = points_gdf.apply(generate_calcite_url, axis=1)

/opt/miniconda3/envs/mapperia_env/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [26]:
points_gdf.to_file("data/four_hearts_ranch_kmz_points.geojson", driver="GeoJSON")

# ALR polygon filter

In [27]:
import geopandas as gpd

# Load ALR polygon data
alr_gdf = gpd.read_file("data/alr_polygon.geojson")


# filter it by Shape_Leng == 2196562.52348
alr_gdf = alr_gdf[alr_gdf['Shape_Leng'] == 2196562.52348]

alr_gdf.to_file("data/alr_polygon_filtered.geojson", driver="GeoJSON")

In [28]:
gdf_with_price

,OBJECTID,GlobalID,Name,PlanID,PlanNumber,PIN,PID,Designation1,Designation2,Designation3,...,ALR status,Assesed Value Land,Assesed Value Improve,Assesed Value TOTAL,BD List Value,Estimated property transfer tax (),Estimated PTT (cultivated farmland),Estimated PTT (rangeland),Four Hearts Package,License
76,77,{911872A1-C142-414B-989C-AA1D7C79A466},011694173,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,6110171.0,11694173,SW4,2957,011-694-173,...,Yes,11855.0,NaN,11855.0,450000.0,9704.0,9704.0,7310.0,FOUR HEARTS RANCH - THE EQUESTRIAN ESTATE,False
1034,1035,{1B4E771F-98A8-40D7-8730-C886A7CD9570},013366599,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,NaN,13366599,NE4,4185,013-366-599,...,Yes,11165.0,NaN,11165.0,400000.0,12080.0,12080.0,9200.0,FOUR HEARTS RANCH - THE 103 MILE RANCH,False
1112,1113,{3EDD98B8-789F-4155-8A81-04F379820023},013312308,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,NaN,13312308,NW4,2951,013-312-308,...,Yes,14825.0,764200.0,779025.0,NaN,28235.0,28235.0,25427.0,FOUR HEARTS RANCH - THE EQUESTRIAN ESTATE,False
3701,3702,{BAE0DA62-0A7B-48D6-97C9-0655DEF0CD49},004293380,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,6133340.0,4293380,722,,004-293-380,...,Yes,5039.0,NaN,5039.0,350000.0,5040.0,5040.0,3600.0,FOUR HEARTS RANCH - THE EQUESTRIAN ESTATE,False
3922,3923,{F1D45DDB-B44F-4123-A469-F073A50D9015},013261771,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,NaN,13261771,720,,013-261-771,...,Yes,7931.0,NaN,7931.0,NaN,9000.0,9000.0,6750.0,FOUR HEARTS RANCH - THE EQUESTRIAN ESTATE,False
4019,4020,{5A6649CD-C879-4F78-9429-F1649FFFE364},003328465,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,6538700.0,3328465,4222,,003-328-465,...,Yes,18375.0,NaN,18375.0,800000.0,12080.0,12080.0,9200.0,THE EXETER PROPERTIES,False
4024,4025,{0715C8CE-14E6-464F-B672-FAC03A348301},013375130,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,NaN,13375130,4224,,013-375-130,...,Yes,4881.0,NaN,4881.0,600000.0,5656.0,5656.0,4090.0,THE MILCH HOMESTEAD RANCH,False
4071,4072,{A4776801-9262-42C2-990A-6749639C7454},013081241,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,6077470.0,13081241,129,,013-081-241,...,Partial,7520.0,NaN,7520.0,600000.0,9200.0,9200.0,9200.0,THE SIMON LAKE RANCH,False
4140,4141,{3BD160A3-814E-4AA5-AD7E-FB1417A0CEAC},011822091,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,6538540.0,11822091,4220,,011-822-091,...,Yes,6696.0,1523000.0,1529696.0,NaN,50478.0,50478.0,45753.0,FOUR HEARTS RANCH - THE EQUESTRIAN ESTATE,False
4168,4169,{8E9247A4-C443-4CD7-93D8-81C5B9B03AB5},007496401,{F0DB3B36-A152-4C45-B118-275779D2B1D7},NO_PLAN,6133210.0,7496401,721,,007-496-401,...,Yes,9760.0,2045800.0,2055560.0,9800000.0,278000.0,49564.0,48484.0,FOUR HEARTS RANCH - THE EQUESTRIAN ESTATE,False
